# This is an exploration notebook used to analyse and understand the sqlite dataset.

**This notebook is structured by exploring loans table seperately and loan_features tabel sepreately, so that the knowledge acquired can be useful when joining tables together during transformation in dbt.**

**Import Packages**

In [404]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import re

**Connection to data**

In [405]:
data_path = Path("../data/YHP_credit_assessment_DS.sqlite")

with sqlite3.connect(data_path) as conn:
    loans = pd.read_sql_query("SELECT * FROM loans", conn)
    features = pd.read_sql_query("SELECT * FROM loan_features", conn)


------

**Data Quality Check for loans table :**

**Column Names**

In [406]:
print(loans.columns)

Index(['loan_id', 'received_date', 'is_loss', 'loss_amount'], dtype='object')


In [407]:
# info about the loans table
loans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   loan_id        10000 non-null  object 
 1   received_date  10000 non-null  object 
 2   is_loss        10000 non-null  int64  
 3   loss_amount    9996 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 312.6+ KB


**Check date range of data**

In [408]:
# Date range of the loans table
print(f"Date range of the loans table: {loans['received_date'].min()} to {loans['received_date'].max()}")

Date range of the loans table: 2018-04-11 to 2025-12-18


**Check for nulls**

In [409]:
# Function to create a null table
def create_null_table(df):
    """Creates a table showing the number and percentage of null values for each column in the DataFrame."""
    null_table = (
        pd.DataFrame({
            "column_name": df.columns,
            "null_count": df.isna().sum().values,
            "null_percentage": (df.isna().mean() * 100).round(2).values,
        })
        .query("null_count > 0")
        .sort_values("null_count", ascending=False)
        .reset_index(drop=True)
    )

    return null_table

In [410]:
# Check for null values in the loans table
display(create_null_table(loans))

,column_name,null_count,null_percentage
0,loss_amount,4,0.04


In [411]:
# Show rows with nulls in the loans table
null_rows_loans = loans[loans.isna().any(axis=1)]
print("Null rows in loans:", len(null_rows_loans))
display(null_rows_loans)

Null rows in loans: 4


,loan_id,received_date,is_loss,loss_amount
661,LRQ-574640,2018-07-13,0,NaN
3391,LRQ-470946,2018-06-03,0,NaN
8134,LRQ-645901,2018-06-03,0,NaN
9417,LRQ-240704,2018-06-03,0,NaN


Check if is_loss is a boolean (0 and 1), and verify if 1 represents loss

In [412]:
# check if the value under is_loss are only 0 and 1
loans["is_loss"].unique()

array([0, 1])

In [413]:
# statistics of loss_amount for loans where is_loss == 0 and is_loss == 1
display(loans[loans["is_loss"] == 0]["loss_amount"].describe())
display(loans[loans["is_loss"] == 1]["loss_amount"].describe())


count     7,877.00
mean        203.40
std       3,322.71
min     -62,560.00
25%           0.00
50%           0.00
75%           0.00
max     101,341.79
Name: loss_amount, dtype: float64

count     2,119.00
mean     31,584.84
std      36,845.20
min      -6,336.10
25%       9,292.19
50%      20,188.87
75%      40,786.71
max     440,332.44
Name: loss_amount, dtype: float64

In [414]:
# check which is_loss values have loss_amount > 0, loss_amount == 0, loss_amount < 0 more than the other

loans["loss_amount_group"] = np.select(
    [
        loans["loss_amount"] > 0,
        loans["loss_amount"] == 0,
        loans["loss_amount"] < 0
    ],
    [
        "positive",
        "zero",
        "negative"
    ],
    default="missing"
)

loss_comparison = pd.crosstab(
    loans["loss_amount_group"],
    loans["is_loss"]
)

loss_comparison.columns = [
    f"is_loss_{int(col)}"
    for col in loss_comparison.columns
]

loss_comparison["higher_count"] = loss_comparison[
    ["is_loss_0", "is_loss_1"]
].idxmax(axis=1)

loss_comparison["difference"] = (
    loss_comparison["is_loss_1"]
    - loss_comparison["is_loss_0"]
)

display(loss_comparison)

,is_loss_0,is_loss_1,higher_count,difference
loss_amount_group,,,,
missing,4,0,is_loss_0,-4
negative,184,13,is_loss_0,-171
positive,133,2099,is_loss_1,1966
zero,7560,7,is_loss_0,-7553


After further inspection, it is verified that is_loss == 0 mean lender did not experience material credit loss and tend to have negative and zero loss_amount.  Whereas, when is_loss == 1, lender experienced a material credit loss or charge-off and have positive loss_amount. Also, null values in loans table are when the lender did not experience loss (0.04% of total dataset). As the task is focudsed on credit loss and risk, it is safe to assume this null values to be 0. 

**Check duplicates**

In [415]:
# Check if loan_id is unique in the loans table
is_unique_loan_id = loans["loan_id"].is_unique
print(f"Is loan_id unique? {is_unique_loan_id}")

Is loan_id unique? True


No duplicates found, so the loan_id is truly unique in loans table.

**Check loan is_loss and when not in is_loss consistency**

In [416]:
# Check if there are any loss loans with zero or negative loss amounts
loss_with_zero_or_negative_amount = loans[
(loans["is_loss"] == 1) & (loans["loss_amount"] <= 0)
]

total_number_of_loss_loans = len(loans[loans["is_loss"] == 1])  

print(
    "Number of loss loans with zero or negative loss:",
    len(loss_with_zero_or_negative_amount)
)

print(
    "Percentage of loss loans with zero or negative loss:",
    round((len(loss_with_zero_or_negative_amount) / total_number_of_loss_loans) * 100, 2),
    "%"
)

display(loss_with_zero_or_negative_amount)

Number of loss loans with zero or negative loss: 20
Percentage of loss loans with zero or negative loss: 0.94 %


,loan_id,received_date,is_loss,loss_amount,loss_amount_group
760,LRQ-790929,2025-08-12,1,-688.31,negative
1059,LRQ-840055,2025-02-12,1,-986.26,negative
1157,LRQ-246071,2025-03-23,1,0.00,zero
1355,LRQ-504561,2025-02-26,1,0.00,zero
2010,LRQ-238261,2025-11-04,1,0.00,zero
3341,LRQ-664222,2024-04-05,1,-147.46,negative
3364,LRQ-283818,2024-12-24,1,"-1,295.78",negative
3401,LRQ-892291,2023-12-09,1,"-1,345.94",negative
3424,LRQ-802531,2025-05-20,1,"-2,279.67",negative
3819,LRQ-340398,2024-12-24,1,"-1,295.78",negative


Only 0.94 % of the data is inconsistent for when SMB is at lost. Few exact figure of loss_amount is being repeated, let's see if there any patterns.

In [417]:
repeated_loss_patterns = (
    loans[
        (loans["is_loss"] == 1) &
        (loans["loss_amount"] <= 0)
    ]
    .groupby(["received_date", "loss_amount"])
    .agg(
        loan_count=("loan_id", "count"),
        loan_ids=("loan_id", lambda x: ", ".join(x))
    )
    .reset_index()
    .query("loan_count > 1")
    .sort_values(
        ["loan_count", "received_date"],
        ascending=[False, True]
    )
)

display(repeated_loss_patterns)

,received_date,loss_amount,loan_count,loan_ids
12,2025-08-12,-688.31,3,"LRQ-790929, LRQ-827813, LRQ-636970"
1,2023-12-09,"-1,345.94",2,"LRQ-892291, LRQ-223433"
5,2024-12-24,"-1,295.78",2,"LRQ-283818, LRQ-340398"
6,2025-02-12,-986.26,2,"LRQ-840055, LRQ-725084"
10,2025-03-23,0.00,2,"LRQ-246071, LRQ-769738"


Several records have exactly same loss amount and received date. This could indicate few things:
- repeated accounting adjustments
- recoveries or reversals processed in batches
- duplicated source values
rather than isolated random error. 

The data dictionary permits occasional negative realised losses, which is why the values are not to be removed. All records will be retained for further analysis.

In [418]:
# Check if there are any not loss loans with positive loss amounts
not_loss_with_positive_amount = loans[
    (loans["is_loss"] == 0) & (loans["loss_amount"] > 0)
]

print(
    "Rows where is_loss == 0 and loss_amount > 0:",
    len(not_loss_with_positive_amount)
)

print(
    "Not loss loans with positive loss percentage:",
    round((len(not_loss_with_positive_amount) / len(loans)) * 100, 2),
    "%"
)

display(not_loss_with_positive_amount)

Rows where is_loss == 0 and loss_amount > 0: 133
Not loss loans with positive loss percentage: 1.33 %


,loan_id,received_date,is_loss,loss_amount,loss_amount_group
3,LRQ-698073,2018-09-01,0,"2,045.16",positive
25,LRQ-494446,2018-10-27,0,"80,982.48",positive
112,LRQ-273189,2019-12-21,0,"25,973.94",positive
127,LRQ-856948,2020-09-27,0,"16,560.00",positive
177,LRQ-283091,2020-09-21,0,"10,914.05",positive
...,...,...,...,...,...
9922,LRQ-946305,2020-11-03,0,"1,106.69",positive
9927,LRQ-634265,2018-12-25,0,"9,867.22",positive
9928,LRQ-690406,2020-03-03,0,"3,442.90",positive
9974,LRQ-311892,2018-08-20,0,"1,028.49",positive


Unlike, above situation, these anamolies, when the lender is not at loss is not acceptable per data dicitonary, so further investigation must be made and should be flagged during analytics in dbt.

In [419]:
non_loss_positive = loans[
    (loans["is_loss"] == 0) &
    (loans["loss_amount"] > 0)
]

display(round(non_loss_positive["loss_amount"].describe(), 2))

count       133.00
mean     14,726.05
std      19,892.41
min           4.62
25%       2,488.56
50%       7,820.00
75%      18,180.12
max     101,341.79
Name: loss_amount, dtype: float64

In [420]:
# Check the quantiles of loss_amount for not loss loans with positive loss amounts
round(non_loss_positive["loss_amount"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]), 2)

0.00         4.62
0.25     2,488.56
0.50     7,820.00
0.75    18,180.12
0.90    39,120.27
0.95    72,288.28
0.99    82,098.88
1.00   101,341.79
Name: loss_amount, dtype: float64

**Distribution Checks**

In [421]:
loans.groupby("is_loss")["loss_amount"].agg(
    count="count",
    missing=lambda x: x.isna().sum(),
    minimum="min",
    median="median",
    mean="mean",
    maximum="max",
)

,count,missing,minimum,median,mean,maximum
is_loss,,,,,,
0,7877,4,"-62,560.00",0.00,203.40,"101,341.79"
1,2119,0,"-6,336.10","20,188.87","31,584.84","440,332.44"


In [422]:
# Display the percentage distribution of loss loans and not loss loans
print("The distibution of SMB lending dataset in loss and not loss categories:")
print(f"Loss loans: {round((loans['is_loss'].sum() / len(loans)) * 100, 2)}%")
print(f"Not loss loans: {round(((len(loans) - loans['is_loss'].sum()) / len(loans)) * 100, 2)}%")

The distibution of SMB lending dataset in loss and not loss categories:
Loss loans: 21.19%
Not loss loans: 78.81%


In [423]:
# Display the percentage of zero loss amount loans, positive loss amount loans and negative loss amount loans
zero_loss_amount_loans = loans[loans["loss_amount"] == 0]
positive_loss_amount_loans = loans[loans["loss_amount"] > 0]
negative_loss_amount_loans = loans[loans["loss_amount"] < 0]

print(f"Zero loss amount loans: {round((len(zero_loss_amount_loans) / len(loans)) * 100, 2)}%")
print(f"Positive loss amount loans: {round((len(positive_loss_amount_loans) / len(loans)) * 100, 2)}%")
print(f"Negative loss amount loans: {round((len(negative_loss_amount_loans) / len(loans)) * 100, 2)}%")

Zero loss amount loans: 75.67%
Positive loss amount loans: 22.32%
Negative loss amount loans: 1.97%


----

**Data Quality Check for loan_features table :**

In [424]:
print(features.columns)

Index(['loan_id', 'feat_001', 'feat_002', 'feat_003', 'feat_004',
       'feat_005_std', 'feat_006', 'feat_007_min', 'feat_008', 'feat_009_min',
       'feat_010', 'feat_011', 'feat_012_trend', 'feat_013_mean',
       'feat_005_cv', 'feat_014_min', 'feat_015', 'feat_016', 'feat_017_sum',
       'feat_018', 'feat_019_max', 'feat_020_sum', 'feat_021_mean',
       'feat_022_sum', 'feat_023_min', 'feat_024_sum', 'feat_025_mean',
       'feat_026_min', 'feat_027_mean', 'feat_028_min', 'feat_029_min',
       'feat_021_trend', 'feat_030_max', 'feat_031_min', 'feat_032_sum',
       'feat_033_max', 'feat_034_mean', 'feat_035_sum', 'feat_036_sum',
       'feat_037_max', 'feat_038_mean', 'feat_039_max', 'feat_040_sum',
       'feat_041', 'feat_042_max', 'feat_014_max', 'feat_043_mean',
       'feat_044_min', 'feat_045_sum', 'feat_046_sum', 'feat_047_mean',
       'feat_048_min', 'feat_049_mean', 'feat_050_min', 'feat_051_sum',
       'feat_052_min', 'feat_053_min', 'feat_009_sum', 'feat_054_min',

In [425]:
# info about the features table
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 100 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   loan_id         10000 non-null  object 
 1   feat_001        10000 non-null  object 
 2   feat_002        9994 non-null   float64
 3   feat_003        10000 non-null  object 
 4   feat_004        10000 non-null  float64
 5   feat_005_std    10000 non-null  float64
 6   feat_006        5842 non-null   float64
 7   feat_007_min    10000 non-null  float64
 8   feat_008        5437 non-null   float64
 9   feat_009_min    10000 non-null  float64
 10  feat_010        7163 non-null   float64
 11  feat_011        10000 non-null  float64
 12  feat_012_trend  9882 non-null   float64
 13  feat_013_mean   9870 non-null   float64
 14  feat_005_cv     10000 non-null  float64
 15  feat_014_min    10000 non-null  float64
 16  feat_015        10000 non-null  float64
 17  feat_016        9791 non-null  

**Check for nulls**

In [426]:
# Check for null values in the features table
display(create_null_table(features))

,column_name,null_count,null_percentage
0,feat_008,4563,45.63
1,feat_006,4158,41.58
2,feat_034_mean,3685,36.85
3,feat_010,2837,28.37
4,feat_018,750,7.50
5,feat_060,750,7.50
6,feat_074_max,411,4.11
7,feat_037_max,411,4.11
8,feat_084,331,3.31
9,feat_016,209,2.09


In [427]:
# Show rows with nulls in the features table
null_rows_features = features[features.isna().any(axis=1)]
print("Null rows in features:", len(null_rows_features))
display(null_rows_features)

Null rows in features: 7119


,loan_id,feat_001,feat_002,feat_003,feat_004,feat_005_std,feat_006,feat_007_min,feat_008,feat_009_min,...,feat_082_min,feat_083_min,feat_084,feat_085_max,feat_005_trend,feat_022_min,feat_086_sum,feat_087_mean,feat_088_mean,feat_055_sum
0,LRQ-235185,IND_183,9.50,ST_41,1.00,"3,188.27",0.00,0.00,NaN,0.75,...,"9,998.00",998.00,71.86,1.00,"-3,726.50","9,998.00",998.00,"99,998.00",10.00,998.00
1,LRQ-499729,IND_107,5.50,ST_46,1.00,"10,946.14",0.38,"126,824.00",NaN,1.00,...,18.00,37.00,62.34,9.00,"3,942.50",2.00,267.00,273.00,10.00,88.00
2,LRQ-711628,IND_055,3.50,ST_07,1.00,"4,439.70",0.35,0.00,138.00,1.00,...,"9,995.00",63.00,60.61,7.00,"4,336.00",19.00,48.00,"1,271.00",10.00,996.00
4,LRQ-985316,IND_107,7.25,ST_19,1.00,"14,674.49",NaN,0.00,28.00,0.77,...,327.00,52.00,64.50,9.00,"8,992.50",173.00,88.00,638.00,10.00,80.00
5,LRQ-331394,IND_018,7.50,ST_47,1.00,"9,593.04",0.49,0.00,166.00,1.00,...,218.00,48.00,81.60,4.00,"-11,749.00","1,320.00",192.00,"10,664.00",10.00,138.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,LRQ-967752,IND_048,0.80,ST_07,1.00,"3,037.94",NaN,0.00,52.00,0.67,...,"9,995.00",75.00,19.05,8.00,"-3,607.00",-2.00,168.00,"6,882.50",10.00,"1,088.00"
9995,LRQ-350203,IND_092,4.16,ST_46,1.00,"17,484.72",NaN,0.00,NaN,0.82,...,"9,995.00",0.00,54.11,7.00,"13,767.00",-20.00,"1,016.00",281.50,10.00,"1,992.00"
9997,LRQ-973820,IND_092,1.00,ST_07,1.00,"106,356.83",0.00,0.00,46.00,0.73,...,"9,997.00",997.00,56.06,1.00,"111,818.50","9,997.00",203.00,"99,997.00",10.00,108.00
9998,LRQ-116150,IND_165,8.66,ST_32,1.00,"13,528.14",NaN,0.00,NaN,0.69,...,200.00,70.00,38.96,6.00,"-15,887.00",5.00,180.00,"1,273.00",10.00,135.00


feat_001, feat_003 and feat_004 seem to be categorical, so further analysis to confirm if it is categorical, and see any patterns/trends.

**Check loan id is unique in feature**

In [428]:
# Check if loan_id is unique in the features table
unique_feature_loan_id = features["loan_id"].is_unique
print(f"Is loan_id unique? {unique_feature_loan_id}")

Is loan_id unique? True


As there way too many columns to analyse and all of them are anonymised (with no data dictionary), firstly, I will try to examine them by grouping with with feat only, feat min, feat max, feat mean, feat sum, feat trend, feat cv and feat std.

In [429]:
suffixes = []

for col in features.columns:
    match = re.match(r"^feat_\d+(?:_(.+))?$", col)

    if match:
        suffix = match.group(1)

        # Columns such as feat_001 have no suffix
        suffixes.append(suffix if suffix is not None else "feature_only")

unique_suffixes = sorted(set(suffixes))

print(unique_suffixes)

['cv', 'feature_only', 'max', 'mean', 'min', 'std', 'sum', 'trend']


In [430]:
suffix_summary = (
    pd.Series(suffixes, name="suffix")
    .value_counts()
    .rename_axis("suffix")
    .reset_index(name="column_count")
)

display(suffix_summary)

,suffix,column_count
0,min,27
1,mean,18
2,sum,16
3,max,16
4,feature_only,14
5,trend,4
6,std,2
7,cv,2


**Group feat by Suffix**

In [431]:
feature_groups = {
    "feature_only": features.filter(regex=r"^feat_\d+$").columns.tolist(),
    "min": features.filter(regex=r"^feat_\d+_min$").columns.tolist(),
    "max": features.filter(regex=r"^feat_\d+_max$").columns.tolist(),
    "mean": features.filter(regex=r"^feat_\d+_mean$").columns.tolist(),
    "sum": features.filter(regex=r"^feat_\d+_sum$").columns.tolist(),
    "trend": features.filter(regex=r"^feat_\d+_trend$").columns.tolist(),
    "cv": features.filter(regex=r"^feat_\d+_cv$").columns.tolist(),
    "std": features.filter(regex=r"^feat_\d+_std$").columns.tolist(),
}

**Feat**

In [432]:
features[feature_groups["feature_only"]].head(10)

,feat_001,feat_002,feat_003,feat_004,feat_006,feat_008,feat_010,feat_011,feat_015,feat_016,feat_018,feat_041,feat_060,feat_084
0,IND_183,9.50,ST_41,1.00,0.00,NaN,676.00,0.03,0.00,0.00,NaN,B,NaN,71.86
1,IND_107,5.50,ST_46,1.00,0.38,NaN,"1,208.58",0.04,0.00,0.00,0.77,A,"84,420.00",62.34
2,IND_055,3.50,ST_07,1.00,0.35,138.00,197.73,0.09,0.00,0.00,0.56,C,"3,750.00",60.61
3,IND_165,17.00,ST_41,0.00,1.20,23.00,90.00,0.21,0.00,"3,410.00",0.54,A,"8,740.00",37.66
4,IND_107,7.25,ST_19,1.00,NaN,28.00,"2,487.00",0.00,0.00,0.00,0.63,A,"18,350.00",64.50
5,IND_018,7.50,ST_47,1.00,0.49,166.00,171.11,0.06,1.00,0.00,0.43,A,"22,500.00",81.60
6,IND_216,10.08,ST_32,1.00,0.73,78.00,164.00,0.22,0.00,832.00,0.49,C,"204,100.00",46.54
7,IND_161,2.08,ST_22,1.00,NaN,39.00,500.00,0.45,0.00,0.00,0.56,B,"13,350.00",54.55
8,IND_064,1.50,ST_33,1.00,NaN,62.00,232.25,0.13,1.00,"10,363.00",NaN,A,NaN,44.59
9,IND_064,12.65,ST_37,1.00,NaN,NaN,2.00,0.01,1.00,"9,976.00",0.88,A,"5,900.00",52.38


In [433]:
# Check the info of the selected feat only columns
features[feature_groups["feature_only"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   feat_001  10000 non-null  object 
 1   feat_002  9994 non-null   float64
 2   feat_003  10000 non-null  object 
 3   feat_004  10000 non-null  float64
 4   feat_006  5842 non-null   float64
 5   feat_008  5437 non-null   float64
 6   feat_010  7163 non-null   float64
 7   feat_011  10000 non-null  float64
 8   feat_015  10000 non-null  float64
 9   feat_016  9791 non-null   float64
 10  feat_018  9250 non-null   float64
 11  feat_041  10000 non-null  object 
 12  feat_060  9250 non-null   float64
 13  feat_084  9669 non-null   float64
dtypes: float64(11), object(3)
memory usage: 1.1+ MB


In [434]:
# Check the percentage of missing values in the feat only columns
features[feature_groups["feature_only"]].isna().mean()*100

feat_001    0.00
feat_002    0.06
feat_003    0.00
feat_004    0.00
feat_006   41.58
feat_008   45.63
feat_010   28.37
feat_011    0.00
feat_015    0.00
feat_016    2.09
feat_018    7.50
feat_041    0.00
feat_060    7.50
feat_084    3.31
dtype: float64

In [435]:
# Ranking each feat only based on the percentage of missing values in descending order
display(create_null_table(features[feature_groups["feature_only"]]))

,column_name,null_count,null_percentage
0,feat_008,4563,45.63
1,feat_006,4158,41.58
2,feat_010,2837,28.37
3,feat_018,750,7.50
4,feat_060,750,7.50
5,feat_084,331,3.31
6,feat_016,209,2.09
7,feat_002,6,0.06


In [436]:
# Display feature only columnns with no missing values
display(
    features[feature_groups["feature_only"]].columns[
        features[feature_groups["feature_only"]].isna().sum() == 0
        ].tolist()
)

['feat_001', 'feat_003', 'feat_004', 'feat_011', 'feat_015', 'feat_041']

**Check feat_001**

In [437]:
def summarise_category(
    dataframe,
    column_name,
    top_n=20,
    include_missing=False
):
    """
    Summarise the values in a categorical column.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        DataFrame containing the column.
    column_name : str
        Name of the column to summarise.
    top_n : int, default=20
        Number of most frequent categories to display.
    include_missing : bool, default=False
        Whether to include missing feature values as a separate group.

    Returns
    -------
    pandas.DataFrame
        Category counts and percentages.
    """

    print("Column:", column_name)
    print("Data type:", dataframe[column_name].dtype)
    print(
        "Unique categories:",
        dataframe[column_name].nunique(dropna=False)
    )

    summary = (
        dataframe[column_name]
        .value_counts(dropna=False) 
        .rename_axis(column_name)
        .reset_index(name="loan_count")
    )

    summary["loan_percentage"] = (
        summary["loan_count"] / len(dataframe) * 100
    ).round(2)

    display(summary.head(top_n))

    return summary

In [438]:
summarise_category(features, "feat_001")

Column: feat_001
Data type: object
Unique categories: 232


,feat_001,loan_count,loan_percentage
0,IND_183,965,9.65
1,IND_064,882,8.82
2,IND_048,511,5.11
3,IND_019,411,4.11
4,IND_020,407,4.07
5,IND_012,324,3.24
6,IND_092,272,2.72
7,IND_018,240,2.40
8,IND_107,230,2.30
9,IND_191,226,2.26


,feat_001,loan_count,loan_percentage
0,IND_183,965,9.65
1,IND_064,882,8.82
2,IND_048,511,5.11
3,IND_019,411,4.11
4,IND_020,407,4.07
...,...,...,...
227,IND_144,1,0.01
228,IND_176,1,0.01
229,IND_177,1,0.01
230,IND_172,1,0.01


In [439]:
def analyse_feature_by_loss(
    loans,
    features,
    feature_name,
    top_n=20,
    include_missing=False
):
    """
    Analyse loan performance by a selected feature.

    Parameters
    ----------
    loans : pandas.DataFrame
        Must contain loan_id, is_loss, and loss_amount.
    features : pandas.DataFrame
        Must contain loan_id and the selected feature.
    feature_name : str
        Name of the feature to analyse.
    top_n : int, default=20
        Number of feature groups to display.
    include_missing : bool, default=False
        Whether to include missing feature values as a separate group.

    Returns
    -------
    pandas.DataFrame
        Full feature-level loss analysis.
    """

    feature_analysis = (
        loans[["loan_id", "is_loss", "loss_amount"]]
        .merge(
            features[["loan_id", feature_name]],
            on="loan_id",
            how="left",
            validate="one_to_one"
        )
        .groupby(
            feature_name,
            dropna=not include_missing
        )
        .agg(
            loan_count=("loan_id", "count"),
            loss_count=("is_loss", "sum"),
            loss_rate=("is_loss", "mean"),
            average_loss_amount=("loss_amount", "mean")
        )
        .reset_index()
    )

    feature_analysis["loss_rate_pct"] = (
        feature_analysis["loss_rate"] * 100
    )

    feature_analysis = feature_analysis.sort_values(
        "loan_count",
        ascending=False
    )

    display(
        feature_analysis
        .head(top_n)
        .round(2)
    )

    return feature_analysis

In [440]:
analyse_feature_by_loss(loans, features, "feat_001", top_n=20, include_missing=True)

,feat_001,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
182,IND_183,965,254,0.26,"8,304.73",26.32
63,IND_064,882,178,0.20,"5,787.23",20.18
47,IND_048,511,92,0.18,"5,490.01",18.00
18,IND_019,411,129,0.31,"16,722.03",31.39
19,IND_020,407,37,0.09,"3,047.84",9.09
11,IND_012,324,68,0.21,"5,108.61",20.99
91,IND_092,272,59,0.22,"5,116.04",21.69
17,IND_018,240,57,0.24,"5,777.17",23.75
106,IND_107,230,47,0.20,"7,190.02",20.43
190,IND_191,226,52,0.23,"6,883.35",23.01


,feat_001,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
182,IND_183,965,254,0.26,"8,304.73",26.32
63,IND_064,882,178,0.20,"5,787.23",20.18
47,IND_048,511,92,0.18,"5,490.01",18.00
18,IND_019,411,129,0.31,"16,722.03",31.39
19,IND_020,407,37,0.09,"3,047.84",9.09
...,...,...,...,...,...,...
191,IND_192,1,0,0.00,0.00,0.00
175,IND_176,1,0,0.00,0.00,0.00
173,IND_174,1,1,1.00,"21,014.24",100.00
171,IND_172,1,1,1.00,"64,778.65",100.00


**Feat 003**

In [441]:
summarise_category(features, "feat_003", top_n=20, include_missing=True)

Column: feat_003
Data type: object
Unique categories: 55


,feat_003,loan_count,loan_percentage
0,ST_32,1346,13.46
1,ST_46,1330,13.30
2,ST_07,982,9.82
3,ST_06,570,5.70
4,ST_50,475,4.75
5,ST_08,444,4.44
6,ST_39,329,3.29
7,ST_41,301,3.01
8,ST_03,296,2.96
9,ST_27,267,2.67


,feat_003,loan_count,loan_percentage
0,ST_32,1346,13.46
1,ST_46,1330,13.30
2,ST_07,982,9.82
3,ST_06,570,5.70
4,ST_50,475,4.75
5,ST_08,444,4.44
6,ST_39,329,3.29
7,ST_41,301,3.01
8,ST_03,296,2.96
9,ST_27,267,2.67


In [442]:
analyse_feature_by_loss(loans, features, "feat_003", top_n=20, include_missing=True)

,feat_003,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
31,ST_32,1346,309,0.23,"9,559.14",22.96
45,ST_46,1330,269,0.20,"5,764.20",20.23
6,ST_07,982,203,0.21,"7,455.16",20.67
5,ST_06,570,112,0.20,"6,622.77",19.65
49,ST_50,475,122,0.26,"7,092.91",25.68
7,ST_08,444,56,0.13,"2,874.61",12.61
38,ST_39,329,68,0.21,"7,531.62",20.67
40,ST_41,301,70,0.23,"6,388.35",23.26
2,ST_03,296,71,0.24,"7,062.24",23.99
26,ST_27,267,51,0.19,"4,915.61",19.10


,feat_003,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
31,ST_32,1346,309,0.23,"9,559.14",22.96
45,ST_46,1330,269,0.20,"5,764.20",20.23
6,ST_07,982,203,0.21,"7,455.16",20.67
5,ST_06,570,112,0.20,"6,622.77",19.65
49,ST_50,475,122,0.26,"7,092.91",25.68
7,ST_08,444,56,0.13,"2,874.61",12.61
38,ST_39,329,68,0.21,"7,531.62",20.67
40,ST_41,301,70,0.23,"6,388.35",23.26
2,ST_03,296,71,0.24,"7,062.24",23.99
26,ST_27,267,51,0.19,"4,915.61",19.10


**Feat 004**

In [443]:
summarise_category(features, "feat_004", include_missing=True)

Column: feat_004
Data type: float64
Unique categories: 2


,feat_004,loan_count,loan_percentage
0,1.00,6362,63.62
1,0.00,3638,36.38


,feat_004,loan_count,loan_percentage
0,1.00,6362,63.62
1,0.00,3638,36.38


In [444]:
analyse_feature_by_loss(loans, features, "feat_004", top_n=20, include_missing=True)

,feat_004,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
1,1.00,6362,1586,0.25,"7,994.30",24.93
0,0.00,3638,533,0.15,"4,865.19",14.65


,feat_004,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
1,1.00,6362,1586,0.25,"7,994.30",24.93
0,0.00,3638,533,0.15,"4,865.19",14.65


**Feat 15**

In [445]:
summarise_category(features, "feat_015", include_missing=True)

Column: feat_015
Data type: float64
Unique categories: 2


,feat_015,loan_count,loan_percentage
0,0.00,5860,58.60
1,1.00,4140,41.40


,feat_015,loan_count,loan_percentage
0,0.00,5860,58.60
1,1.00,4140,41.40


In [446]:
analyse_feature_by_loss(loans, features, "feat_015", top_n=20, include_missing=False)

,feat_015,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,0.00,5860,1367,0.23,"7,663.78",23.33
1,1.00,4140,752,0.18,"5,711.00",18.16


,feat_015,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,0.00,5860,1367,0.23,"7,663.78",23.33
1,1.00,4140,752,0.18,"5,711.00",18.16


**Feat 41**

In [447]:
summarise_category(features, "feat_041", include_missing=True)

Column: feat_041
Data type: object
Unique categories: 4


,feat_041,loan_count,loan_percentage
0,A,5307,53.07
1,B,3165,31.65
2,C,1488,14.88
3,D,40,0.40


,feat_041,loan_count,loan_percentage
0,A,5307,53.07
1,B,3165,31.65
2,C,1488,14.88
3,D,40,0.40


In [448]:
analyse_feature_by_loss(loans, features, "feat_041", top_n=20, include_missing=False)

,feat_041,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,A,5307,1197,0.23,"6,856.63",22.56
1,B,3165,616,0.19,"7,839.58",19.46
2,C,1488,299,0.20,"4,910.67",20.09
3,D,40,7,0.18,"1,333.94",17.50


,feat_041,loan_count,loss_count,loss_rate,average_loss_amount,loss_rate_pct
0,A,5307,1197,0.23,"6,856.63",22.56
1,B,3165,616,0.19,"7,839.58",19.46
2,C,1488,299,0.20,"4,910.67",20.09
3,D,40,7,0.17,"1,333.94",17.50


**feat with min**

In [449]:
features[feature_groups["min"]].head(10)

,feat_007_min,feat_009_min,feat_014_min,feat_023_min,feat_026_min,feat_028_min,feat_029_min,feat_031_min,feat_044_min,feat_048_min,...,feat_066_min,feat_067_min,feat_068_min,feat_070_min,feat_071_min,feat_072_min,feat_057_min,feat_082_min,feat_083_min,feat_022_min
0,0.00,0.75,418.00,"51,021.00",998.00,"99,998.00",998.00,998.00,"9,996.00","99,998.00",...,999.00,55.00,998.00,996.00,"25,563.03","9,998.00",1.30,"9,998.00",998.00,"9,998.00"
1,"126,824.00",1.00,303.00,"35,586.00",-1.00,893.00,14.00,9.00,0.00,"2,300.00",...,999.00,80.00,-8.00,996.00,0.85,-32.00,11.48,18.00,37.00,2.00
2,0.00,1.00,27.00,800.00,6.00,405.00,-64.00,14.00,42.00,13.00,...,999.00,29.00,-22.00,996.00,0.06,"9,995.00",1.68,"9,995.00",63.00,19.00
3,426.00,0.70,411.00,"99,500.00",-3.00,125.00,-54.00,-17.00,"9,996.00",693.00,...,999.00,201.00,24.00,2.00,0.13,-66.00,8.87,0.00,37.00,"9,994.00"
4,0.00,0.77,279.00,"1,000.00",1.00,171.00,0.00,4.00,-6.00,317.00,...,65.00,164.00,-1.00,996.00,0.16,-5.00,2.00,327.00,52.00,173.00
5,0.00,1.00,453.00,"67,257.00",0.00,"8,105.00",-102.00,-32.00,24.00,92.00,...,999.00,45.00,61.00,996.00,0.39,35.00,2.90,218.00,48.00,"1,320.00"
6,0.00,0.90,415.00,800.00,0.00,"1,343.00",-12.00,-10.00,30.00,"2,982.00",...,999.00,360.00,-10.00,996.00,0.31,"9,995.00",7.64,"9,996.00",45.00,-100.00
7,"15,586.00",0.94,259.00,"2,000.00",13.00,"1,419.00",-17.00,-10.00,-1.00,209.00,...,999.00,45.00,23.00,27.00,0.23,-49.00,1.42,-64.00,21.00,-36.00
8,"1,000,000,000.00",0.33,415.00,"1,000,000,000.00",998.00,"99,998.00",998.00,998.00,"9,997.00","99,998.00",...,999.00,23.00,998.00,996.00,"39,178.81","9,998.00",0.20,"9,998.00",998.00,"9,998.00"
9,0.00,0.76,178.00,"5,000.00",0.00,463.00,-5.00,-4.00,26.00,372.00,...,999.00,73.00,-4.00,62.00,0.05,120.00,2.25,-10.00,57.00,-5.00


In [450]:
# Check the first 10 rows of the features table,
# splitting the columns into two halves for easier viewing

midpoint = len(features[feature_groups["min"]].columns) // 2

first_half_columns = features[feature_groups["min"]].columns[:midpoint]
second_half_columns = features[feature_groups["min"]].columns[midpoint:]

display(features[feature_groups["min"]].loc[:, first_half_columns].head(10))
display(features[feature_groups["min"]].loc[:, second_half_columns].head(10))


,feat_007_min,feat_009_min,feat_014_min,feat_023_min,feat_026_min,feat_028_min,feat_029_min,feat_031_min,feat_044_min,feat_048_min,feat_050_min,feat_052_min,feat_053_min
0,0.00,0.75,418.00,"51,021.00",998.00,"99,998.00",998.00,998.00,"9,996.00","99,998.00","50,533.00","1,000,000,000.00",998.00
1,"126,824.00",1.00,303.00,"35,586.00",-1.00,893.00,14.00,9.00,0.00,"2,300.00","166,795.00",50.00,0.00
2,0.00,1.00,27.00,800.00,6.00,405.00,-64.00,14.00,42.00,13.00,"1,000,000,000.00",297.00,998.00
3,426.00,0.70,411.00,"99,500.00",-3.00,125.00,-54.00,-17.00,"9,996.00",693.00,"1,000,000,000.00",88.00,264.00
4,0.00,0.77,279.00,"1,000.00",1.00,171.00,0.00,4.00,-6.00,317.00,"1,000,000,000.00",77.00,6.00
5,0.00,1.00,453.00,"67,257.00",0.00,"8,105.00",-102.00,-32.00,24.00,92.00,"1,000,000,000.00",417.00,998.00
6,0.00,0.90,415.00,800.00,0.00,"1,343.00",-12.00,-10.00,30.00,"2,982.00","1,000,000,000.00",77.00,3.00
7,"15,586.00",0.94,259.00,"2,000.00",13.00,"1,419.00",-17.00,-10.00,-1.00,209.00,128.00,982.00,0.00
8,"1,000,000,000.00",0.33,415.00,"1,000,000,000.00",998.00,"99,998.00",998.00,998.00,"9,997.00","99,998.00","1,000,000,000.00","1,000,000,000.00",998.00
9,0.00,0.76,178.00,"5,000.00",0.00,463.00,-5.00,-4.00,26.00,372.00,"1,000,000,000.00",130.00,998.00


,feat_054_min,feat_055_min,feat_058_min,feat_062_min,feat_066_min,feat_067_min,feat_068_min,feat_070_min,feat_071_min,feat_072_min,feat_057_min,feat_082_min,feat_083_min,feat_022_min
0,"9,998.00",998.00,"9,998.00","1,000,000,000.00",999.00,55.00,998.00,996.00,"25,563.03","9,998.00",1.30,"9,998.00",998.00,"9,998.00"
1,-79.00,88.00,-868.00,"532,114.00",999.00,80.00,-8.00,996.00,0.85,-32.00,11.48,18.00,37.00,2.00
2,"9,995.00",996.00,-15.00,"5,579.00",999.00,29.00,-22.00,996.00,0.06,"9,995.00",1.68,"9,995.00",63.00,19.00
3,0.00,71.00,-23.00,"96,445.00",999.00,201.00,24.00,2.00,0.13,-66.00,8.87,0.00,37.00,"9,994.00"
4,47.00,80.00,-5.00,"1,000,000,000.00",65.00,164.00,-1.00,996.00,0.16,-5.00,2.00,327.00,52.00,173.00
5,218.00,46.00,-216.00,"1,000,000,000.00",999.00,45.00,61.00,996.00,0.39,35.00,2.90,218.00,48.00,"1,320.00"
6,"9,996.00",159.00,-35.00,"1,000,000,000.00",999.00,360.00,-10.00,996.00,0.31,"9,995.00",7.64,"9,996.00",45.00,-100.00
7,-67.00,39.00,-25.00,"1,000,000,000.00",999.00,45.00,23.00,27.00,0.23,-49.00,1.42,-64.00,21.00,-36.00
8,"9,998.00",998.00,"9,998.00","1,000,000,000.00",999.00,23.00,998.00,996.00,"39,178.81","9,998.00",0.20,"9,998.00",998.00,"9,998.00"
9,-10.00,125.00,-13.00,"1,000,000,000.00",999.00,73.00,-4.00,62.00,0.05,120.00,2.25,-10.00,57.00,-5.00


**feat min info**

In [464]:
pd.set_option("display.float_format", "{:,.2f}".format) # improve readability for data with power (e)

In [465]:
def summarise_feature_group(features, feature_groups, group_name):
    """Summarise a named group of feature columns."""
    feature_columns = feature_groups[group_name]
    selected_features = features[feature_columns]

    summary = (
        selected_features
        .agg(["count", "min", "max", "median", "mean", "std"])
        .T
        .reset_index()
        .rename(columns={"index": "feature"})
    )

    summary["null_count"] = selected_features.isna().sum().to_numpy()
    summary["null_percentage"] = (
        summary["null_count"] / len(selected_features) * 100
    )

    return summary

In [471]:
display(summarise_feature_group(features,feature_groups,"min"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_007_min,"10,000.00",0.00,"1,000,000,000.00",870.00,"200,219,216.41","400,160,324.74",0,0.00
1,feat_009_min,"10,000.00",0.00,10.00,0.86,0.81,0.74,0,0.00
2,feat_014_min,"10,000.00",0.00,780.00,271.00,262.64,131.07,0,0.00
3,feat_023_min,"10,000.00",0.00,"1,000,000,000.00","4,655.50","42,725,268.91","202,184,687.84",0,0.00
4,feat_026_min,"9,870.00",-17.00,999.00,2.00,160.41,363.43,130,1.30
5,feat_028_min,"9,870.00",0.00,"99,999.00","1,169.00","24,831.91","41,617.10",130,1.30
6,feat_029_min,"9,870.00",-200.00,999.00,-2.00,223.27,434.82,130,1.30
7,feat_031_min,"9,870.00",-148.00,999.00,5.00,190.00,392.03,130,1.30
8,feat_044_min,"9,870.00","-9,992.00","9,999.00",22.00,"4,096.99","4,907.35",130,1.30
9,feat_048_min,"9,870.00","-6,027.00","99,999.00",497.00,"13,331.15","32,135.82",130,1.30


Several patterns of 999, 9999, 99999 and 1000000000. As they are unusually neat boundary values they may represent:
- missing or unavailable
- "no limit" or "not applicable"
- capped values
- default code system
- upper boundaries

In [467]:
min_features = features[feature_groups["min"]]

display(min_features[min_features.isna().any(axis=1)])

,feat_007_min,feat_009_min,feat_014_min,feat_023_min,feat_026_min,feat_028_min,feat_029_min,feat_031_min,feat_044_min,feat_048_min,...,feat_066_min,feat_067_min,feat_068_min,feat_070_min,feat_071_min,feat_072_min,feat_057_min,feat_082_min,feat_083_min,feat_022_min
38,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
41,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
66,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
70,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
120,"1,000,000,000.00",10.00,305.00,"1,000,000,000.00",NaN,NaN,NaN,NaN,NaN,NaN,...,999.00,999.00,NaN,999.00,"60,906.28",NaN,"60,906.28",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9771,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
9807,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
9857,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
9957,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN


In [454]:
min_columns = feature_groups["min"]

display(
    features.loc[
        features[min_columns].isna().any(axis=1),
        ["loan_id", *min_columns]
    ]
)

,loan_id,feat_007_min,feat_009_min,feat_014_min,feat_023_min,feat_026_min,feat_028_min,feat_029_min,feat_031_min,feat_044_min,...,feat_066_min,feat_067_min,feat_068_min,feat_070_min,feat_071_min,feat_072_min,feat_057_min,feat_082_min,feat_083_min,feat_022_min
38,LRQ-156007,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
41,LRQ-880360,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
66,LRQ-205945,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
70,LRQ-629258,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
120,LRQ-305226,"1,000,000,000.00",10.00,305.00,"1,000,000,000.00",NaN,NaN,NaN,NaN,NaN,...,999.00,999.00,NaN,999.00,"60,906.28",NaN,"60,906.28",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9771,LRQ-288004,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
9807,LRQ-842978,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
9857,LRQ-930206,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN
9957,LRQ-897901,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,...,0.00,0.00,NaN,0.00,0.00,NaN,0.00,NaN,NaN,NaN


**Null paterns are identical or not?**

In [455]:
min_columns = feature_groups["min"]

columns_with_nulls = [
    column
    for column in min_columns
    if features[column].isna().any()
]

null_pattern = features[columns_with_nulls].isna()

print("Columns with nulls:", len(columns_with_nulls))
print("Rows with at least one null:", null_pattern.any(axis=1).sum())
print("Rows where all nullable columns are null:", null_pattern.all(axis=1).sum())

Columns with nulls: 14
Rows with at least one null: 130
Rows where all nullable columns are null: 130


**Missingness pattern**

In [457]:
features["has_missing_feature_group"] = (
    features[columns_with_nulls]
    .isna()
    .any(axis=1)
    .astype(int)
)

missing_group_analysis = (
    loans[["loan_id", "is_loss", "loss_amount"]]
    .merge(
        features[["loan_id", "has_missing_feature_group"]],
        on="loan_id",
        how="left"
    )
    .groupby("has_missing_feature_group")
    .agg(
        loan_count=("loan_id", "count"),
        loss_count=("is_loss", "sum"),
        loss_rate=("is_loss", "mean"),
        average_loss_amount=("loss_amount", "mean"),
        median_loss_amount=("loss_amount", "median")
    )
    .reset_index()
)

missing_group_analysis["loss_rate_pct"] = (
    missing_group_analysis["loss_rate"] * 100
)

display(missing_group_analysis.round(2))

,has_missing_feature_group,loan_count,loss_count,loss_rate,average_loss_amount,median_loss_amount,loss_rate_pct
0,0,9870,2098,0.21,"6,884.52",0.00,21.26
1,1,130,21,0.16,"4,675.02",0.00,16.15


**Sentinal Value Analysis**

In [ ]:
suspected_sentinel_values = [999, 9_999, 99_999, 1_000_000_000]

sentinel_summary = []

for column in min_features.columns:
    for value in suspected_sentinel_values:
        count = min_features[column].eq(value).sum()

        if count > 0:
            sentinel_summary.append(
                {
                    "feature": column,
                    "value": value,
                    "count": count,
                    "percentage": count / len(min_features) * 100
                }
            )

sentinel_summary = pd.DataFrame(sentinel_summary)

display(
    sentinel_summary
    .sort_values(["percentage", "feature"], ascending=[False, True])
    .style.format(
        {
            "value": "{:,.2f}",
            "count": "{:,.0f}",
            "percentage": "{:.2f}%"
        }
    )
)

,feature,value,count,percentage
17,feat_066_min,999.00,"8,693",86.93%
10,feat_050_min,"1,000,000,000.00","7,247",72.47%
16,feat_062_min,"1,000,000,000.00","6,698",66.98%
11,feat_052_min,"1,000,000,000.00","2,384",23.84%
0,feat_007_min,"1,000,000,000.00","2,002",20.02%
1,feat_023_min,"1,000,000,000.00",427,4.27%
25,feat_022_min,"9,999.00",104,1.04%
2,feat_026_min,999.00,104,1.04%
4,feat_028_min,"99,999.00",104,1.04%
5,feat_029_min,999.00,104,1.04%


**Date checks for feature min**

In [ ]:
missing_group_dates = (
    loans[["loan_id", "received_date"]]
    .merge(
        features[["loan_id", "has_missing_feature_group"]],
        on="loan_id",
        how="left"
    )
    .groupby("has_missing_feature_group")
    .agg(
        loan_count=("loan_id", "count"),
        earliest_date=("received_date", "min"),
        latest_date=("received_date", "max")
    )
    .reset_index()
)

display(missing_group_dates)

,has_missing_feature_group,loan_count,earliest_date,latest_date
0,0,9870,2018-04-11,2025-12-18
1,1,130,2018-04-24,2025-12-05


**feat max**

In [458]:
features[feature_groups["max"]].head(10)

,feat_019_max,feat_030_max,feat_033_max,feat_037_max,feat_039_max,feat_042_max,feat_014_max,feat_059_max,feat_061_max,feat_064_max,feat_074_max,feat_076_max,feat_077_max,feat_078_max,feat_081_max,feat_085_max
0,"99,998.00","1,000,000,000.00","9,998.00",790.00,2.00,998.00,418.00,"99,998.00","36,714.00","99,995.00",0.07,"9,996.00",2.00,998.00,4.00,1.00
1,-174.00,"1,000,000,000.00","9,996.00","11,966.00",13.00,30.00,303.00,"14,994.00","102,889.00","99,995.00",1.68,22.00,80.00,51.00,95.00,9.00
2,-20.00,269.00,609.00,"1,057.00",8.00,998.00,27.00,267.00,"10,027.00","99,995.00",0.82,117.00,14.00,93.00,8.00,7.00
3,165.00,"1,000,000,000.00",-40.00,"5,144.00",11.00,35.00,411.00,695.00,"36,380.00","-6,089.00",2.11,"9,996.00",201.00,59.00,998.00,6.00
4,60.00,"1,000,000,000.00",17.00,"2,041.00",12.00,30.00,279.00,0.00,"21,500.00","-5,776.00",0.16,6.00,46.00,56.00,295.00,9.00
5,-100.00,"1,000,000,000.00",-28.00,"1,394.00",8.00,998.00,453.00,0.00,"28,345.00","99,995.00",0.40,247.00,72.00,63.00,404.00,4.00
6,"6,203.00","1,000,000,000.00",267.00,"1,385.00",7.00,34.00,415.00,"2,797.00","91,028.00","-5,016.00",0.08,113.00,121.00,43.00,998.00,4.00
7,54.00,"1,639.00","1,342.00","1,069.00",13.00,996.00,259.00,403.00,"11,449.00","99,995.00",0.32,-7.00,26.00,17.00,50.00,9.00
8,"99,998.00","1,000,000,000.00","9,998.00",NaN,2.00,998.00,415.00,"99,998.00","3,043.00","99,997.00",NaN,"9,997.00",106.00,998.00,558.00,0.00
9,478.00,"1,000,000,000.00",180.00,"3,038.00",7.00,998.00,178.00,0.00,"87,480.00","99,995.00",0.23,31.00,160.00,97.00,998.00,2.00


**feat max info**

In [472]:
display(summarise_feature_group(features,feature_groups,"max"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_019_max,"9,870.00","-21,517.00","99,999.00",56.00,"21,055.59","40,586.68",130,1.30
1,feat_030_max,"10,000.00",0.00,"1,000,000,000.00","1,000,000,000.00","728,500,622.12","444,754,566.39",0,0.00
2,feat_033_max,"9,870.00","-9,992.00","9,999.00","9,995.00","5,037.05","5,182.19",130,1.30
3,feat_037_max,"9,589.00",0.00,"100,000,000.00","1,820.00","13,342.49","1,021,187.41",411,4.11
4,feat_039_max,"10,000.00",0.00,99.00,10.00,11.89,11.79,0,0.00
5,feat_042_max,"9,870.00",0.00,999.00,996.00,654.39,451.88,130,1.30
6,feat_014_max,"10,000.00",0.00,780.00,285.00,279.20,124.13,0,0.00
7,feat_059_max,"9,870.00",0.00,"99,999.00",693.00,"19,914.40","38,584.65",130,1.30
8,feat_061_max,"10,000.00",0.00,"1,000,000,000.00","29,665.50","68,750,677.91","252,942,191.04",0,0.00
9,feat_064_max,"9,870.00","-70,605.00","99,999.00","99,995.00","73,862.59","47,199.55",130,1.30


In [461]:
max_features = features[feature_groups["max"]]

max_value_frequency = []

for column in max_features.columns:
    column_max = max_features[column].max()
    count_at_max = max_features[column].eq(column_max).sum()

    max_value_frequency.append(
        {
            "feature": column,
            "maximum_value": column_max,
            "count_at_max": count_at_max,
            "percentage_at_max": count_at_max / len(max_features) * 100
        }
    )

max_value_frequency = pd.DataFrame(max_value_frequency)

display(
    max_value_frequency
    .sort_values("percentage_at_max", ascending=False)
    .style.format(
        {
            "maximum_value": "{:,.2f}",
            "count_at_max": "{:,.0f}",
            "percentage_at_max": "{:.2f}%"
        }
    )
)

,feature,maximum_value,count_at_max,percentage_at_max
1,feat_030_max,"1,000,000,000.00","7,285",72.85%
8,feat_061_max,"1,000,000,000.00",687,6.87%
2,feat_033_max,"9,999.00",132,1.32%
0,feat_019_max,"99,999.00",132,1.32%
7,feat_059_max,"99,999.00",132,1.32%
11,feat_076_max,"9,999.00",132,1.32%
9,feat_064_max,"99,999.00",132,1.32%
5,feat_042_max,999.00,132,1.32%
13,feat_078_max,999.00,132,1.32%
15,feat_085_max,99.00,55,0.55%


Several _max features display strong boundary effects, with medians at or close to values such as 999, 9,999, 99,999, and 1 billion. Other features are extremely right-skewed, with means substantially above their medians. In addition, groups of 130 and 411 loans share consistent missingness patterns. These findings suggest possible feature caps, sentinel values, or shared upstream data dependencies just like in feature min that should be investigated before modelling.

**feat mean**

In [462]:
features[feature_groups["mean"]].head(10)

,feat_013_mean,feat_021_mean,feat_025_mean,feat_027_mean,feat_034_mean,feat_038_mean,feat_043_mean,feat_047_mean,feat_049_mean,feat_056_mean,feat_014_mean,feat_065_mean,feat_069_mean,feat_073_mean,feat_079_mean,feat_080_mean,feat_087_mean,feat_088_mean
0,"9,998.00",8.33,998.00,2.00,NaN,998.00,37.00,"99,998.00",998.00,998.00,418.00,3.00,2.00,10.00,1.00,0.00,"99,998.00",10.00
1,"9,996.00",18.00,998.00,3.00,"12,004.00",267.00,3.00,704.00,278.00,-11.00,303.00,16.00,14.00,0.55,2.00,0.46,273.00,10.00
2,98.00,26.33,998.00,9.00,NaN,23.00,11.00,4.00,-2.00,-81.00,27.00,4.00,12.00,0.41,98.00,0.00,"1,271.00",10.00
3,60.00,22.00,998.00,138.00,"3,579.00",164.00,24.00,278.00,3.00,-17.00,411.00,5.00,9.00,0.17,0.00,0.67,491.00,10.00
4,13.00,8.67,998.00,4.00,"8,794.00",49.00,6.00,232.00,8.00,-9.00,279.00,3.00,21.00,0.79,1.00,0.29,638.00,10.00
5,-6.00,6.00,998.00,26.00,NaN,64.00,23.00,53.00,94.00,0.00,453.00,8.00,11.00,0.64,0.00,0.62,"10,664.00",10.00
6,-20.00,7.67,998.00,16.00,"6,772.00",48.00,16.00,"1,544.00",28.00,-7.00,415.00,3.00,16.00,0.06,0.00,0.50,"3,193.00",10.00
7,"9,995.00",10.67,998.00,4.00,"6,738.00",50.00,2.00,178.00,24.00,-57.00,259.00,0.00,45.00,0.46,1.00,0.00,"2,139.00",10.00
8,"9,998.00",23.33,998.00,106.00,NaN,3.00,75.00,"99,998.00",998.00,998.00,415.00,0.00,996.00,10.00,0.00,1.00,"99,998.00",10.00
9,138.00,62.33,19.00,81.00,"4,736.00",237.00,68.00,164.00,-22.00,-2.00,178.00,4.00,81.00,0.86,0.00,0.78,"1,451.00",10.00


In [473]:
display(summarise_feature_group(features,feature_groups,"mean"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_013_mean,"9,870.00",-100.00,"9,999.00","9,995.00","5,834.69","4,736.52",130,1.30
1,feat_021_mean,"10,000.00",0.00,"4,725.67",19.33,29.66,101.02,0,0.00
2,feat_025_mean,"10,000.00",0.00,999.00,998.00,797.00,380.38,0,0.00
3,feat_027_mean,"10,000.00",0.00,999.00,13.00,72.37,206.07,0,0.00
4,feat_034_mean,"6,315.00",0.00,"520,328.00","3,412.00","7,120.95","19,455.99",3685,36.85
5,feat_038_mean,"10,000.00",0.00,999.00,88.00,133.35,162.71,0,0.00
6,feat_043_mean,"10,000.00",0.00,999.00,12.00,39.40,127.17,0,0.00
7,feat_047_mean,"9,870.00","-10,161.00","99,999.00",294.00,"13,752.02","32,713.93",130,1.30
8,feat_049_mean,"9,870.00",-992.00,999.00,6.00,118.77,354.90,130,1.30
9,feat_056_mean,"9,870.00",-172.00,999.00,-2.00,219.88,417.08,130,1.30


feat_034_mean has exponentially higher null values compared to other feat means.

In [476]:
feat_034_missing_analysis = (
    loans[["loan_id", "received_date", "is_loss", "loss_amount"]]
    .merge(
        features[["loan_id", "feat_034_mean"]],
        on="loan_id",
        how="left"
    )
    .assign(
        feat_034_missing=lambda df: df["feat_034_mean"].isna()
    )
    .groupby("feat_034_missing")
    .agg(
        loan_count=("loan_id", "count"),
        loss_count=("is_loss", "sum"),
        loss_rate=("is_loss", "mean"),
        median_loss_amount=("loss_amount", "median"),
        earliest_date=("received_date", "min"),
        latest_date=("received_date", "max")
    )
    .reset_index()
)

feat_034_missing_analysis["loss_rate_pct"] = (
    feat_034_missing_analysis["loss_rate"] * 100
)

display(feat_034_missing_analysis.round(2))

,feat_034_missing,loan_count,loss_count,loss_rate,median_loss_amount,earliest_date,latest_date,loss_rate_pct
0,False,6315,1312,0.21,0.00,2018-04-27,2025-12-15,20.78
1,True,3685,807,0.22,0.00,2018-04-11,2025-12-18,21.90


In [475]:
mean_features = features[feature_groups["mean"]]

mean_value_frequency = []

for column in mean_features.columns:
    column_mean = mean_features[column].mean()
    count_at_mean = mean_features[column].eq(column_max).sum()

    mean_value_frequency.append(
        {
            "feature": column,
            "mean_value": column_mean,
            "count_at_mean": count_at_mean,
            "percentage_at_mean": count_at_mean / len(mean_features) * 100
        }
    )

mean_value_frequency = pd.DataFrame(mean_value_frequency)

display(
    mean_value_frequency
    .sort_values("percentage_at_mean", ascending=False)
    .style.format(
        {
            "mean_value": "{:,.2f}",
            "count_at_mean": "{:,.0f}",
            "percentage_at_mean": "{:.2f}%"
        }
    )
)

,feature,mean_value,count_at_mean,percentage_at_mean
5,feat_038_mean,133.35,39,0.39%
11,feat_065_mean,11.57,29,0.29%
14,feat_079_mean,32.99,29,0.29%
12,feat_069_mean,84.69,12,0.12%
10,feat_014_mean,270.97,8,0.08%
3,feat_027_mean,72.37,7,0.07%
9,feat_056_mean,219.88,5,0.05%
7,feat_047_mean,"13,752.02",3,0.03%
8,feat_049_mean,118.77,3,0.03%
6,feat_043_mean,39.40,2,0.02%


The _mean feature group contains a mix of relatively stable variables and several features with strong boundary effects. Features such as feat_013_mean, feat_025_mean, and feat_088_mean are concentrated close to their apparent upper limits, while feat_034_mean, feat_047_mean, and feat_087_mean are strongly right-skewed. Missingness is structured rather than isolated: five features are jointly missing for 130 loans, and feat_034_mean is unavailable for 36.85% of the portfolio. These patterns suggest possible capping, sentinel values, or source-specific availability and should be tested before modelling rather than treated as ordinary continuous values

**feat sum**

In [477]:
features[feature_groups["sum"]].head(20)

,feat_017_sum,feat_020_sum,feat_022_sum,feat_024_sum,feat_032_sum,feat_035_sum,feat_036_sum,feat_040_sum,feat_045_sum,feat_046_sum,feat_051_sum,feat_009_sum,feat_038_sum,feat_057_sum,feat_086_sum,feat_055_sum
0,"1,000,000,000.00",42.00,"9,998.00","9,998.00",55.00,27.89,0.25,92.00,0.00,"1,000,000,000.00","2,043.00",0.75,998.00,1.30,998.00,998.00
1,"1,000,000,000.00",996.00,2.00,"9,996.00",80.00,93.53,0.27,8.00,"39,971.00","6,008.00",459.00,1.00,267.00,11.48,267.00,88.00
2,297.00,996.00,19.00,120.00,4.00,52.50,0.75,11.00,"22,012.00",270.00,943.00,1.00,23.00,1.68,48.00,996.00
3,"2,000,000,000.00",12.00,"19,988.00","19,992.00",402.00,73.84,0.20,74.00,0.00,928.00,"4,556.00",1.40,328.00,17.75,276.00,142.00
4,216.00,996.00,173.00,-36.00,164.00,49.50,0.23,18.00,"11,231.00",618.00,"1,876.00",0.77,49.00,2.00,88.00,80.00
5,"1,251.00","2,988.00","3,960.00",-288.00,135.00,133.05,0.40,78.00,0.00,"8,568.00","-28,545.00",3.00,192.00,8.71,192.00,138.00
6,"1,000,000,000.00",173.00,-100.00,"9,995.00",360.00,62.21,0.20,35.00,0.00,"19,052.00","-2,299.00",0.90,48.00,7.64,16.00,159.00
7,"4,532.00",18.00,-36.00,-32.00,13.00,37.71,0.64,4.00,734.00,993.00,"1,574.00",0.94,50.00,1.42,50.00,39.00
8,"1,000,000,000.00",2.00,"9,998.00","9,998.00",23.00,34.38,0.00,106.00,0.00,"1,000,000,000.00",0.00,0.33,3.00,0.20,998.00,998.00
9,430.00,42.00,-5.00,126.00,237.00,4.97,0.00,81.00,0.00,341.00,"2,694.00",0.76,237.00,2.25,261.00,125.00


In [478]:
display(summarise_feature_group(features,feature_groups,"sum"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_017_sum,"10,000.00",0.00,"4,000,000,256.00","2,658.00","469,800,801.96","575,082,934.58",0,0.00
1,feat_020_sum,"10,000.00",0.00,"6,972.00",26.00,315.67,501.52,0,0.00
2,feat_022_sum,"10,000.00",-300.00,"39,992.00",118.00,"3,468.32","5,062.53",0,0.00
3,feat_024_sum,"10,000.00",-288.00,"69,972.00","9,995.00","7,211.73","5,878.99",0,0.00
4,feat_032_sum,"10,000.00",0.00,"3,157.00",80.00,162.03,250.39,0,0.00
5,feat_035_sum,"10,000.00",0.00,"11,952.80",47.26,62.30,146.33,0,0.00
6,feat_036_sum,"10,000.00",0.00,11.75,0.20,0.39,1.07,0,0.00
7,feat_040_sum,"10,000.00",0.00,"2,009.00",21.00,78.84,215.78,0,0.00
8,feat_045_sum,"10,000.00",0.00,"3,000,000,000.00","10,096.50","168,032,884.37","400,477,923.66",0,0.00
9,feat_046_sum,"10,000.00","-16,080.00","4,000,000,000.00","1,018.50","88,301,907.56","300,519,311.30",0,0.00


**feat std**

In [479]:
features[feature_groups["std"]].head(20)

,feat_005_std,feat_021_std
0,"3,188.27",0.94
1,"10,946.14",0.82
2,"4,439.70",8.38
3,184.50,3.00
4,"14,674.49",1.25
5,"9,593.04",1.41
6,"7,218.26",0.47
7,"3,295.87",2.05
8,"1,966.71",1.25
9,"61,079.81",8.18


In [480]:
display(summarise_feature_group(features,feature_groups,"std"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_005_std,"10,000.00",0.00,"1,253,419.88","8,197.97","19,595.62","45,042.85",0,0.00
1,feat_021_std,"10,000.00",0.00,"6,663.30",2.83,6.96,117.13,0,0.00


**feat trend**

In [481]:
features[feature_groups["trend"]].head(20)

,feat_012_trend,feat_021_trend,feat_075_trend,feat_005_trend
0,"-4,878.50",-1.00,0.00,"-3,726.50"
1,"1,526.00",-0.50,0.00,"3,942.50"
2,"5,416.00",-7.00,-0.50,"4,336.00"
3,"-6,160.00",-6.00,0.00,-369.00
4,"8,992.50",-1.50,0.00,"8,992.50"
5,"-12,999.00",-1.50,0.00,"-11,749.00"
6,"11,925.00",0.50,0.00,-262.50
7,"-2,903.00",-2.50,0.00,"1,523.50"
8,"-8,049.00",-0.50,1.50,"-2,404.00"
9,"61,767.00",-10.00,-4.00,"61,905.50"


In [485]:
display(summarise_feature_group(features,feature_groups,"trend"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_012_trend,"9,882.00","-815,665.00","1,165,743.00",-76.50,-18.12,"49,772.96",118,1.18
1,feat_021_trend,"9,882.00","-1,098.00","7,067.50",0.00,2.20,125.12,118,1.18
2,feat_075_trend,"9,882.00",-39.00,38.50,0.00,0.00,1.59,118,1.18
3,feat_005_trend,"9,882.00","-768,143.00","508,825.50",34.75,-334.37,"40,576.66",118,1.18


**feat cv**

In [486]:
features[feature_groups["cv"]].head(20)

,feat_005_cv,feat_063_cv
0,0.08,0.24
1,0.23,0.25
2,0.31,0.56
3,0.02,0.14
4,0.37,0.39
5,0.41,0.50
6,0.25,0.29
7,0.20,0.40
8,0.08,0.53
9,0.61,0.51


In [484]:
display(summarise_feature_group(features,feature_groups,"cv"))

,feature,count,min,max,median,mean,std,null_count,null_percentage
0,feat_005_cv,"10,000.00",0.00,1.41,0.21,0.26,0.19,0,0.00
1,feat_063_cv,"10,000.00",-18.99,9.69,0.28,0.32,0.33,0,0.00


----

In [ ]:
missingness_results = []

for col in features.columns:
    if col == "loan_id" or features[col].isna().sum() == 0:
        continue

    temp = (
        features[["loan_id", col]]
        .merge(
            loans[["loan_id", "is_loss", "loss_amount"]],
            on="loan_id",
            how="inner"
        )
    )

    temp["is_missing"] = temp[col].isna()

    summary = (
        temp.groupby("is_missing")
        .agg(
            loan_count=("loan_id", "count"),
            loss_rate=("is_loss", "mean"),
            average_loss_amount=("loss_amount", "mean")
        )
    )

    missing_loss_rate = summary.loc[True, "loss_rate"]
    present_loss_rate = summary.loc[False, "loss_rate"]

    missingness_results.append({
        "feature": col,
        "missing_count": temp["is_missing"].sum(),
        "missing_pct": temp["is_missing"].mean() * 100,
        "loss_rate_when_missing": missing_loss_rate * 100,
        "loss_rate_when_present": present_loss_rate * 100,
        "loss_rate_difference": (missing_loss_rate - present_loss_rate) * 100
    })

missingness_analysis = (
    pd.DataFrame(missingness_results)
    .sort_values(
        "loss_rate_difference",
        key=abs,
        ascending=False
    )
)

display(round(missingness_analysis, 2))

,feature,missing_count,missing_pct,loss_rate_when_missing,loss_rate_when_present,loss_rate_difference
0,feat_002,6,0.06,33.33,21.18,12.15
6,feat_016,209,2.09,28.71,21.03,7.68
1,feat_006,4158,41.58,24.77,18.64,6.13
3,feat_010,2837,28.37,25.31,19.56,5.75
5,feat_013_mean,130,1.30,16.15,21.26,-5.10
8,feat_019_max,130,1.30,16.15,21.26,-5.10
11,feat_029_min,130,1.30,16.15,21.26,-5.10
10,feat_028_min,130,1.30,16.15,21.26,-5.10
9,feat_026_min,130,1.30,16.15,21.26,-5.10
40,feat_087_mean,130,1.30,16.15,21.26,-5.10


In [ ]:
data = features.merge(
    loans[["loan_id", "received_date", "is_loss"]],
    on="loan_id",
    how="inner"
)

data["received_year"] = pd.to_datetime(
    data["received_date"]
).dt.year

high_missing_features = [
    "feat_008",
    "feat_006",
    "feat_034_mean",
    "feat_010"
]

missing_by_year = (
    data.groupby("received_year")[high_missing_features]
    .apply(lambda frame: frame.isna().mean() * 100)
)

display(missing_by_year.round(2))

,feat_008,feat_006,feat_034_mean,feat_010
received_year,,,,
2018,63.67,44.68,43.42,33.29
2019,63.28,36.07,43.52,29.48
2020,41.19,31.14,36.35,24.19
2021,38.82,30.26,36.84,20.39
2022,33.69,43.52,41.98,28.66
2023,44.37,44.70,38.50,32.79
2024,46.32,42.87,30.55,25.64
2025,42.97,41.34,28.11,23.63
